In [1]:
from pathlib import Path
import csv
import math
import struct

import dpkt
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import DBSCAN, KMeans, SpectralClustering
import hdbscan


## Filtering, subsampling, and clustering

In [2]:
def prepare_points(points, min_distance, max_distance, keep_fraction, seed, box=None):
    # Reduce number of points analysed to improve computation time
    distances = np.linalg.norm(points, axis=1)
    filtered = points[(distances >= min_distance) & (distances <= max_distance)]
    if box is not None:
        filtered = filtered[box_mask(filtered, box)]
    if len(filtered) and keep_fraction < 1.0:
        keep_count = max(1, int(np.ceil(len(filtered) * keep_fraction)))
        rng = np.random.default_rng(seed)
        indices = np.sort(rng.choice(len(filtered), size=keep_count, replace=False))
        filtered = filtered[indices]
    return filtered


def cluster_points(points, method, cluster_count, dbscan_eps, min_samples, hdbscan_min_cluster_size, spectral_neighbors):
    # Cluster points with specified method and parameters

    point_count = len(points)
    if point_count == 0:
        return np.empty(0, dtype=int)
    if method == 'K-means':
        return KMeans(n_clusters=min(cluster_count, point_count), n_init=10, random_state=42).fit_predict(points)
    if method == 'Spectral':
        if point_count < 2:
            return np.zeros(point_count, dtype=int)
        count = min(cluster_count, point_count - 1)
        neighbors = min(spectral_neighbors, point_count - 1)
        return SpectralClustering(n_clusters=count, affinity='nearest_neighbors', n_neighbors=neighbors, assign_labels='kmeans', random_state=42).fit_predict(points)
    if method == 'DBSCAN':
        return DBSCAN(eps=dbscan_eps, min_samples=min_samples).fit_predict(points)
    if method == 'HDBSCAN':
        if point_count < 2:
            return np.full(point_count, -1, dtype=int)
        return hdbscan.HDBSCAN(min_cluster_size=min(hdbscan_min_cluster_size, point_count), min_samples=min_samples).fit_predict(points)
    raise ValueError(f'Unknown clustering method: {method}')


def cluster_frame(points, method, settings, seed):
    selected = prepare_points(points, settings['min_distance'], settings['max_distance'], settings['keep_fraction'], seed, settings.get('box_filter'))
    labels = cluster_points(selected, method, settings['cluster_count'], settings['dbscan_eps'], settings['min_samples'], settings['hdbscan_min_cluster_size'], settings['spectral_neighbors'])
    cluster_labels = set(labels.tolist()) - {-1}
    return {
        'points': selected,
        'labels': labels,
        'decoded_count': len(points),
        'used_count': len(selected),
        'cluster_count': len(cluster_labels),
        'noise_count': int(np.count_nonzero(labels == -1)),
    }

## Open file

Set up file to analyse

In [15]:
# File to analyse 
PCAP_PATH = Path("../data/driveway.pcap")

# Angle correction file used by Jt128 sensor
CALIBRATION_PATH = Path("../Lidar setup/JT128_angle_correction.csv")

# Frames to work with while clustering
FIRST_FRAME = 2
LAST_FRAMES = 100


Load file as to analyse for using angle correction from JT128 sensor

In [16]:

def load_calibration(path):
    calibration = {}
    with path.open("r", encoding="utf-8", errors="ignore", newline="") as calibration_file:
        for row in csv.DictReader(calibration_file):
            try:
                channel = int(row["Channel"])
                elevation = float(row["Elevation"])
                azimuth = float(row["Azimuth"])
            except (KeyError, TypeError, ValueError):
                continue
            calibration[channel] = (elevation, azimuth)

    if sorted(calibration) != list(range(1, 129)):
        raise ValueError(f"Expected 128 JT128 calibration rows in {path}.")

    elevations = np.array([calibration[channel][0] for channel in range(1, 129)], dtype=float)
    azimuth_offsets = np.array([calibration[channel][1] for channel in range(1, 129)], dtype=float)
    return elevations, azimuth_offsets


VERTICAL_ANGLES_DEG, AZIMUTH_OFFSETS_DEG = load_calibration(CALIBRATION_PATH)


def decode_packet(payload):
    """ Decode file as per Jt128 sensor """
    if len(payload) < 12 or payload[:4] != b"\xee\xff\x01\x04":
        return []

    laser_count, block_count, _, distance_unit_mm, _, status = payload[6:12]
    if laser_count != len(VERTICAL_ANGLES_DEG):
        raise ValueError(f"Packet has {laser_count} channels, but the calibration has {len(VERTICAL_ANGLES_DEG)}.")

    unit_size = 3 + bool(status & 0x10) + bool(status & 0x20) + bool(status & 0x40)
    block_size = 2 + laser_count * unit_size
    optional_size = 4  # CRC following the block data
    optional_size += 17 if status & 0x04 else 0  # Function-safety data
    optional_size += 26  # Hesai packet tail
    optional_size += 4 if status & 0x01 else 0  # Sequence number
    optional_size += 22 if status & 0x02 else 0  # IMU data
    optional_size += 4  # Packet CRC
    optional_size += 32 if status & 0x08 else 0  # Cyber-security signature
    expected_size = 12 + block_count * block_size + optional_size
    if len(payload) != expected_size:
        raise ValueError(f"Unexpected Hesai v1.4 payload size {len(payload)}; expected {expected_size}.")

    distance_unit_m = distance_unit_mm / 1000.0
    decoded_blocks = []
    block_offset = 12
    for _ in range(block_count):
        azimuth_deg = int.from_bytes(payload[block_offset:block_offset + 2], "little") / 100.0
        block_offset += 2
        azimuths = np.radians((azimuth_deg + AZIMUTH_OFFSETS_DEG) % 360.0)
        elevations = np.radians(VERTICAL_ANGLES_DEG)
        distances = np.empty(laser_count, dtype=float)

        for channel in range(laser_count):
            record = payload[block_offset:block_offset + unit_size]
            raw_distance = struct.unpack_from("<H", record)[0]
            distances[channel] = raw_distance * distance_unit_m
            block_offset += unit_size

        horizontal = distances * np.cos(elevations)
        points = np.column_stack((
            horizontal * np.sin(azimuths),
            horizontal * np.cos(azimuths),
            distances * np.sin(elevations),
        ))
        points = points[distances > 0]
        decoded_blocks.append((azimuth_deg, points))

    return decoded_blocks


frames = []
frame_blocks = []
previous_azimuth = None

with PCAP_PATH.open("rb") as capture_file:
    reader = dpkt.pcap.Reader(capture_file)
    if reader.datalink() != dpkt.pcap.DLT_EN10MB:
        raise ValueError("Expected an Ethernet .pcap capture.")

    for _, raw_packet in reader:
        try:
            ethernet = dpkt.ethernet.Ethernet(raw_packet)
            network = ethernet.data
            if isinstance(network, dpkt.ip.IP):
                transport = network.data
            elif isinstance(network, dpkt.ip6.IP6):
                transport = network.data
            else:
                continue
            if not isinstance(transport, dpkt.udp.UDP):
                continue
            decoded_blocks = decode_packet(bytes(transport.data))
        except (AttributeError, dpkt.dpkt.NeedData, ValueError):
            continue

        for azimuth, points in decoded_blocks:
            if previous_azimuth is not None and previous_azimuth - azimuth > 180:
                if frame_blocks:
                    frames.append(np.vstack(frame_blocks))
                    frame_blocks = []
                if len(frames) >= LAST_FRAMES:
                    break
            if len(points):
                frame_blocks.append(points)
            previous_azimuth = azimuth

        if len(frames) >= LAST_FRAMES:
            break

if len(frames) < LAST_FRAMES and frame_blocks:
    frames.append(np.vstack(frame_blocks))
if not frames:
    raise ValueError("No valid Hesai protocol v1.4 LiDAR frames found in the capture.")

first_frame = FIRST_FRAME
print(f"Loaded {len(frames)} frame(s) (From {FIRST_FRAME} to {LAST_FRAMES}) from {PCAP_PATH.resolve()}")
print("Points per frame: " + ", ".join(f"{len(frame):,}" for frame in frames))

Loaded 100 frame(s) (From 2 to 100) from C:\Users\danag\Documents\LIDAR\data\driveway.pcap
Points per frame: 65,038, 98,062, 98,074, 98,063, 98,029, 98,170, 98,560, 98,870, 99,626, 100,016, 100,889, 100,702, 101,105, 101,875, 103,470, 104,938, 106,402, 106,764, 107,443, 107,744, 107,417, 107,620, 107,576, 107,671, 107,446, 108,587, 110,540, 110,879, 110,577, 109,734, 110,936, 111,893, 109,822, 109,492, 107,165, 104,982, 105,967, 105,908, 105,230, 104,691, 104,493, 103,875, 103,255, 102,618, 101,848, 101,318, 100,927, 100,576, 100,404, 99,986, 99,559, 99,343, 95,082, 99,146, 98,818, 98,621, 98,463, 98,280, 98,385, 98,409, 98,211, 98,341, 98,176, 98,193, 98,056, 98,044, 97,904, 97,901, 97,809, 98,121, 98,082, 97,932, 97,955, 97,815, 97,998, 97,826, 98,042, 97,917, 97,912, 97,932, 97,921, 97,863, 98,032, 97,889, 97,791, 98,011, 97,889, 97,698, 97,738, 97,850, 97,901, 97,847, 97,770, 97,810, 97,947, 97,794, 97,756, 97,721, 97,898, 97,704


## Plot frames in sequence

In [17]:
from IPython.display import clear_output, display
import ipywidgets as widgets

PLOT_MIN_DISTANCE_M = 0.0
PLOT_MAX_DISTANCE_M = 3.0
PLOT_KEEP_FRACTION = 0.8
PLOT_RANDOM_SEED = 42

method_widget = widgets.Dropdown(
    options=["K-means", "DBSCAN", "HDBSCAN", "Spectral"],
    value="DBSCAN",
    description="Method",
)
frame_count_widget = widgets.IntSlider(
    value=min(LAST_FRAMES, len(frames)),
    min=FIRST_FRAME,
    max=LAST_FRAMES,
    step=1,
    description="Frames",
    continuous_update=False,
)
min_distance_widget = widgets.FloatText(value=PLOT_MIN_DISTANCE_M, description="Min range (m)")
max_distance_widget = widgets.FloatText(value=PLOT_MAX_DISTANCE_M, description="Max range (m)")
keep_fraction_widget = widgets.FloatSlider(
    value=PLOT_KEEP_FRACTION,
    min=0.05,
    max=1.0,
    step=0.05,
    description="Point fraction",
    readout_format=".2f",
    continuous_update=False,
)
cluster_count_widget = widgets.IntSlider(
    value=5,
    min=2,
    max=30,
    step=1,
    description="K-means/Spectral clusters",
    continuous_update=False,
)
dbscan_eps_widget = widgets.FloatText(value=0.1, description="DBSCAN eps (m)")
min_samples_widget = widgets.IntSlider(
    value=10,
    min=1,
    max=100,
    step=1,
    description="Min samples",
    continuous_update=False,
)
hdbscan_min_cluster_size_widget = widgets.IntSlider(
    value=10,
    min=2,
    max=100,
    description="HDBSCAN min cluster size",
    continuous_update=False,
)
spectral_neighbors_widget = widgets.IntSlider(
    value=10,
    min=1,
    max=100,
    step=1,
    description="Spectral neighbors",
    continuous_update=False,
)
random_seed_widget = widgets.IntText(value=PLOT_RANDOM_SEED, description="Sample seed")
association_gate_widget = widgets.FloatText(
    value=0.75, description="Match gate (m)"
)

scene_minimum = np.min([frame.min(axis=0) for frame in frames], axis=0)
scene_maximum = np.max([frame.max(axis=0) for frame in frames], axis=0)
bbox_xmin_widget = widgets.FloatText(value=float(scene_minimum[0]), description="X min (m)")
bbox_xmax_widget = widgets.FloatText(value=float(scene_maximum[0]), description="X max (m)")
bbox_ymin_widget = widgets.FloatText(value=float(scene_minimum[1]), description="Y min (m)")
bbox_ymax_widget = widgets.FloatText(value=float(scene_maximum[1]), description="Y max (m)")
bbox_zmin_widget = widgets.FloatText(value=float(scene_minimum[2]), description="Z min (m)")
bbox_zmax_widget = widgets.FloatText(value=float(scene_maximum[2]), description="Z max (m)")
use_filter_box_widget = widgets.Checkbox(
    value=False, description="Keep points inside filter box"
)
filter_xmin_widget = widgets.FloatText(value=float(scene_minimum[0]), description="X min (m)")
filter_xmax_widget = widgets.FloatText(value=float(scene_maximum[0]), description="X max (m)")
filter_ymin_widget = widgets.FloatText(value=float(scene_minimum[1]), description="Y min (m)")
filter_ymax_widget = widgets.FloatText(value=float(scene_maximum[1]), description="Y max (m)")
filter_zmin_widget = widgets.FloatText(value=float(scene_minimum[2]), description="Z min (m)")
filter_zmax_widget = widgets.FloatText(value=float(scene_maximum[2]), description="Z max (m)")

run_button = widgets.Button(description="Cluster and plot", button_style="primary", icon="play")
plot_output = widgets.Output()

controls = widgets.VBox([
    widgets.HBox([method_widget, frame_count_widget]),
    widgets.HBox([min_distance_widget, max_distance_widget, keep_fraction_widget]),
    widgets.HBox([cluster_count_widget, dbscan_eps_widget, min_samples_widget]),
    widgets.HBox([
        hdbscan_min_cluster_size_widget,
        spectral_neighbors_widget,
        random_seed_widget,
        association_gate_widget,
    ]),
    widgets.HTML("<b>Manual bounding box</b>"),
    widgets.HBox([bbox_xmin_widget, bbox_xmax_widget]),
    widgets.HBox([bbox_ymin_widget, bbox_ymax_widget]),
    widgets.HBox([bbox_zmin_widget, bbox_zmax_widget]),
    widgets.HTML("<b>Optional point filter box (applied before clustering)</b>"),
    use_filter_box_widget,
    widgets.HBox([filter_xmin_widget, filter_xmax_widget]),
    widgets.HBox([filter_ymin_widget, filter_ymax_widget]),
    widgets.HBox([filter_zmin_widget, filter_zmax_widget]),
])


def box_mask(points, box):
    lower = np.array([box["xmin"], box["ymin"], box["zmin"]])
    upper = np.array([box["xmax"], box["ymax"], box["zmax"]])
    return np.all((points >= lower) & (points <= upper), axis=1)


def get_point_filter_box():
    if not use_filter_box_widget.value:
        return None
    box = {
        "xmin": filter_xmin_widget.value,
        "xmax": filter_xmax_widget.value,
        "ymin": filter_ymin_widget.value,
        "ymax": filter_ymax_widget.value,
        "zmin": filter_zmin_widget.value,
        "zmax": filter_zmax_widget.value,
    }
    if any(box[f"{axis}min"] > box[f"{axis}max"] for axis in ("x", "y", "z")):
        raise ValueError("Each point filter box minimum must be less than or equal to its maximum.")
    return box


def best_cluster_for_box(labels, manual_mask):
    candidates = [label for label in np.unique(labels) if label != -1]
    if not candidates:
        candidates = list(np.unique(labels))

    best = None
    for label in candidates:
        predicted = labels == label
        true_positive = int(np.count_nonzero(predicted & manual_mask))
        false_positive = int(np.count_nonzero(predicted & ~manual_mask))
        false_negative = int(np.count_nonzero(~predicted & manual_mask))
        precision = true_positive / max(true_positive + false_positive, 1)
        recall = true_positive / max(true_positive + false_negative, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-12)
        candidate = {
            "label": label,
            "mask": predicted,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }
        if best is None or candidate["f1"] > best["f1"]:
            best = candidate
    return best


def match_cluster_centroids(points, labels, previous_centroids, association_gate, next_track_id):
    centroids = {
        label: points[labels == label].mean(axis=0)
        for label in np.unique(labels)
        if label != -1 and np.any(labels == label)
    }
    candidate_pairs = sorted(
        (
            float(np.linalg.norm(previous_position - centroids[label])),
            track_id,
            label,
        )
        for track_id, previous_position in previous_centroids.items()
        for label in centroids
    )
    assignments = {}
    used_tracks = set()
    for distance, track_id, label in candidate_pairs:
        if distance <= association_gate and track_id not in used_tracks and label not in assignments:
            assignments[label] = track_id
            used_tracks.add(track_id)

    current_centroids = {}
    for label, centroid in centroids.items():
        if label not in assignments:
            assignments[label] = next_track_id
            next_track_id += 1
        current_centroids[assignments[label]] = centroid.copy()
    return centroids, assignments, current_centroids, next_track_id


def draw_box_edges(axes, box):
    x_values = [box["xmin"], box["xmax"]]
    y_values = [box["ymin"], box["ymax"]]
    z_values = [box["zmin"], box["zmax"]]
    for x_value in x_values:
        for y_value in y_values:
            axes.plot([x_value, x_value], [y_value, y_value], z_values, color="blue", linewidth=1)
    for x_value in x_values:
        for z_value in z_values:
            axes.plot([x_value, x_value], y_values, [z_value, z_value], color="blue", linewidth=1)
    for y_value in y_values:
        for z_value in z_values:
            axes.plot(x_values, [y_value, y_value], [z_value, z_value], color="blue", linewidth=1)


def plot_selected_frames(_):
    run_button.disabled = True
    try:
        box = {
            "xmin": bbox_xmin_widget.value,
            "xmax": bbox_xmax_widget.value,
            "ymin": bbox_ymin_widget.value,
            "ymax": bbox_ymax_widget.value,
            "zmin": bbox_zmin_widget.value,
            "zmax": bbox_zmax_widget.value,
        }
        if any(box[f"{axis}min"] > box[f"{axis}max"] for axis in ("x", "y", "z")):
            raise ValueError("Each bounding-box minimum must be less than or equal to its maximum.")
        association_gate = association_gate_widget.value
        if association_gate <= 0:
            raise ValueError("The centroid matching gate must be positive.")
        point_filter_box = get_point_filter_box()

        with plot_output:
            clear_output(wait=True)
            selected_frames = frames[:frame_count_widget.value]
            frame_numbers = []
            frame_f1_scores = []
            previous_centroids = {}
            next_track_id = 1
            centroid_tracks = {}
            best_cluster_motion_rows = []

            for frame_number, frame in enumerate(selected_frames, start=1):
                plot_points = prepare_points(
                    frame,
                    min_distance=min_distance_widget.value,
                    max_distance=max_distance_widget.value,
                    keep_fraction=keep_fraction_widget.value,
                    seed=random_seed_widget.value + frame_number - 1,
                    box=point_filter_box,
                )
                if not len(plot_points):
                    print(f"Frame {frame_number}/{len(selected_frames)}: no points remain after filtering; F1 = 0.000")
                    frame_numbers.append(frame_number)
                    frame_f1_scores.append(0.0)
                    best_cluster_motion_rows.append({
                        "frame": frame_number,
                        "label": None,
                        "track_id": None,
                        "centroid": None,
                        "displacement": None,
                        "f1": 0.0,
                    })
                    previous_centroids = {}
                    continue

                labels = cluster_points(
                    plot_points,
                    method_widget.value,
                    cluster_count_widget.value,
                    dbscan_eps_widget.value,
                    min_samples_widget.value,
                    hdbscan_min_cluster_size_widget.value,
                    spectral_neighbors_widget.value,
                )
                centroids, track_assignments, current_centroids, next_track_id = match_cluster_centroids(
                    plot_points,
                    labels,
                    previous_centroids,
                    association_gate,
                    next_track_id,
                )
                track_shifts = {}
                track_displacements = {}
                for label, centroid in centroids.items():
                    track_id = track_assignments[label]
                    previous_position = previous_centroids.get(track_id)
                    displacement = (
                        centroid - previous_position
                        if previous_position is not None else np.full(3, np.nan)
                    )
                    shift = float(np.linalg.norm(displacement))
                    track_shifts[label] = shift
                    track_displacements[label] = displacement
                    centroid_tracks.setdefault(track_id, []).append({
                        "frame": frame_number,
                        "position": centroid.copy(),
                        "shift": shift,
                    })
                previous_centroids = current_centroids

                manual_mask = box_mask(plot_points, box)
                best = best_cluster_for_box(labels, manual_mask)
                if best is None:
                    frame_f1 = 0.0
                    print(f"Frame {frame_number}/{len(selected_frames)}: clustering returned no labels; F1 = 0.000")
                    frame_numbers.append(frame_number)
                    frame_f1_scores.append(frame_f1)
                    best_cluster_motion_rows.append({
                        "frame": frame_number,
                        "label": None,
                        "track_id": None,
                        "centroid": None,
                        "displacement": None,
                        "f1": frame_f1,
                    })
                    continue

                frame_f1 = best["f1"]
                frame_numbers.append(frame_number)
                frame_f1_scores.append(frame_f1)
                best_label = best["label"]
                best_cluster_motion_rows.append({
                    "frame": frame_number,
                    "label": int(best_label),
                    "track_id": track_assignments.get(best_label),
                    "centroid": centroids.get(best_label),
                    "displacement": track_displacements.get(best_label),
                    "f1": frame_f1,
                })
                cluster_count = len(set(labels.tolist()) - {-1})
                noise_count = int(np.count_nonzero(labels == -1))
                filter_text = " inside filter box" if point_filter_box is not None else ""
                print(
                    f"Frame {frame_number}/{len(selected_frames)}: "
                    f"{len(plot_points):,} points{filter_text}, {cluster_count} clusters, "
                    f"{noise_count:,} noise points, best label {best['label']}, "
                    f"precision {best['precision']:.3f}, recall {best['recall']:.3f}, F1 {frame_f1:.3f}"
                )
                for label, centroid in centroids.items():
                    track_id = track_assignments[label]
                    shift = track_shifts[label]
                    shift_text = f"{shift:.3f} m/frame" if np.isfinite(shift) else "new track"
                    print(
                        f"  Track {track_id} (cluster {label}): centroid "
                        f"[{centroid[0]:.3f}, {centroid[1]:.3f}, {centroid[2]:.3f}] m; "
                        f"shift {shift_text}"
                    )

                true_positive = best["mask"] & manual_mask
                false_positive = best["mask"] & ~manual_mask
                false_negative = ~best["mask"] & manual_mask
                background = ~best["mask"] & ~manual_mask

                figure = plt.figure(figsize=(10, 8))
                axes = figure.add_subplot(111, projection="3d")
                axes.scatter(*plot_points[background].T, c="lightgray", s=1, alpha=0.25)
                axes.scatter(*plot_points[false_negative].T, c="blue", s=4, label="Box points missed")
                axes.scatter(*plot_points[false_positive].T, c="red", s=4, label="Cluster outside box")
                axes.scatter(*plot_points[true_positive].T, c="green", s=4, label="Cluster inside box")
                for label, centroid in centroids.items():
                    axes.scatter(
                        *centroid, marker="*", s=130, label=f"Track {track_assignments[label]} centroid"
                    )
                    axes.text(*centroid, f"T{track_assignments[label]}", fontsize=8)
                draw_box_edges(axes, box)
                axes.set_title(
                    f"Hesai JT128 frame {frame_number} ({method_widget.value}), "
                    f"best label {best['label']}, F1 {frame_f1:.3f}"
                )
                axes.set_xlabel("X (m)")
                axes.set_ylabel("Y (m)")
                axes.set_zlabel("Z (m)")
                axes.set_box_aspect(np.maximum(np.ptp(plot_points, axis=0), 1e-6))
                axes.legend(loc="upper right")
                plt.tight_layout()
                plt.show()
                plt.close(figure)

            motion_headers = [
                "Frame", "Cluster label", "Track ID", "Centroid X (m)",
                "Centroid Y (m)", "Centroid Z (m)", "Delta X (m/frame)",
                "Delta Y (m/frame)", "Delta Z (m/frame)",
                "Motion (m/frame)", "Box-match F1",
            ]
            motion_table = "<h4>Best cluster motion by frame</h4><table><tr>" + "".join(
                f"<th style='padding:4px 8px'>{header}</th>" for header in motion_headers
            ) + "</tr>"
            for row in best_cluster_motion_rows:
                if row["centroid"] is None:
                    values = [
                        row["frame"], "No matched cluster", "-", "-", "-", "-",
                        "-", "-", "-", "-", f"{row['f1']:.3f}",
                    ]
                else:
                    centroid = row["centroid"]
                    displacement = row["displacement"]
                    values = [
                        row["frame"], row["label"], row["track_id"],
                        *(f"{value:.3f}" for value in centroid),
                        *(f"{value:.3f}" if np.isfinite(value) else "-" for value in displacement),
                        f"{np.linalg.norm(displacement):.3f}"
                        if np.all(np.isfinite(displacement)) else "-",
                        f"{row['f1']:.3f}",
                    ]
                motion_table += "<tr>" + "".join(
                    f"<td style='padding:4px 8px'>{value}</td>" for value in values
                ) + "</tr>"
            motion_table += "</table>"
            display(widgets.HTML(value=motion_table))

            if centroid_tracks:
                figure = plt.figure(figsize=(10, 8))
                axes = figure.add_subplot(111, projection="3d")
                colors = plt.get_cmap("tab10")
                for track_index, (track_id, observations) in enumerate(centroid_tracks.items()):
                    positions = np.asarray([item["position"] for item in observations])
                    observed_frames = [item["frame"] for item in observations]
                    color = colors(track_index % 10)
                    axes.plot(*positions.T, marker="o", color=color, label=f"Track {track_id}")
                    for position, observed_frame in zip(positions, observed_frames):
                        axes.text(*position, str(observed_frame), fontsize=8)
                axes.set_title(f"Cluster centroid trajectories ({method_widget.value})")
                axes.set_xlabel("X (m)")
                axes.set_ylabel("Y (m)")
                axes.set_zlabel("Z (m)")
                axes.legend(loc="upper right")
                plt.tight_layout()
                plt.show()
                plt.close(figure)

                figure, axes = plt.subplots(figsize=(10, 4))
                has_displacement_data = False
                for track_index, (track_id, observations) in enumerate(centroid_tracks.items()):
                    measured = [item for item in observations if np.isfinite(item["shift"])]
                    if measured:
                        has_displacement_data = True
                        axes.plot(
                            [item["frame"] for item in measured],
                            [item["shift"] for item in measured],
                            marker="o",
                            color=colors(track_index % 10),
                            label=f"Track {track_id}",
                        )
                axes.set_title("Cluster centroid displacement by frame")
                axes.set_xlabel("Frame number")
                axes.set_ylabel("Centroid displacement (m/frame)")
                axes.grid(True, alpha=0.3)
                if has_displacement_data:
                    axes.legend(loc="best")
                else:
                    axes.text(
                        0.5, 0.5, "No clusters matched across consecutive frames.",
                        ha="center", va="center", transform=axes.transAxes,
                    )
                plt.tight_layout()
                plt.show()
                plt.close(figure)
            else:
                print("No cluster centroids were available to track.")

            figure, axes = plt.subplots(figsize=(10, 4))
            axes.plot(frame_numbers, frame_f1_scores, marker="o", linewidth=1.5)
            axes.set_title(f"Best cluster F1 by frame ({method_widget.value})")
            axes.set_xlabel("Frame number")
            axes.set_ylabel("F1 score")
            axes.set_ylim(0, 1)
            axes.set_xticks(frame_numbers)
            axes.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
            plt.close(figure)
    except Exception as error:
        with plot_output:
            print(f"Clustering failed: {error}")
    finally:
        run_button.disabled = False


run_button.on_click(plot_selected_frames)
display(controls, run_button, plot_output)

Button(button_style='primary', description='Cluster and plot', icon='play', style=ButtonStyle())

Output()

In [18]:
%matplotlib widget

from IPython.display import clear_output, display
import ipywidgets as widgets

single_frame_widget = widgets.IntSlider(
    value=1,
    min=1,
    max=len(frames),
    step=1,
    description="Frame",
    continuous_update=False,
)
single_frame_button = widgets.Button(description="Show frame", button_style="primary", icon="eye")
single_frame_output = widgets.Output()
SINGLE_FRAME_POINT_LIMIT = 30000


def plot_single_frame(_):
    single_frame_button.disabled = True
    try:
        frame_number = single_frame_widget.value
        frame = frames[frame_number - 1]
        point_filter_box = get_point_filter_box()
        points_to_plot = prepare_points(
            frame,
            min_distance=min_distance_widget.value,
            max_distance=max_distance_widget.value,
            keep_fraction=keep_fraction_widget.value,
            seed=random_seed_widget.value + frame_number - 1,
            box=point_filter_box,
        )
        if not len(points_to_plot):
            raise ValueError("No points remain after applying the current distance and sampling settings.")

        box = {
            "xmin": bbox_xmin_widget.value,
            "xmax": bbox_xmax_widget.value,
            "ymin": bbox_ymin_widget.value,
            "ymax": bbox_ymax_widget.value,
            "zmin": bbox_zmin_widget.value,
            "zmax": bbox_zmax_widget.value,
        }
        if any(box[f"{axis}min"] > box[f"{axis}max"] for axis in ("x", "y", "z")):
            raise ValueError("Each bounding-box minimum must be less than or equal to its maximum.")

        inside_box = box_mask(points_to_plot, box)
        inside_points = points_to_plot[inside_box]
        outside_points = points_to_plot[~inside_box]
        rng = np.random.default_rng(random_seed_widget.value + frame_number - 1)
        if len(inside_points) >= SINGLE_FRAME_POINT_LIMIT:
            inside_points = inside_points[rng.choice(len(inside_points), SINGLE_FRAME_POINT_LIMIT, replace=False)]
            outside_points = np.empty((0, 3))
        else:
            outside_limit = SINGLE_FRAME_POINT_LIMIT - len(inside_points)
            if len(outside_points) > outside_limit:
                outside_points = outside_points[rng.choice(len(outside_points), outside_limit, replace=False)]

        with single_frame_output:
            clear_output(wait=True)
            figure = plt.figure(figsize=(10, 8))
            axes = figure.add_subplot(111, projection="3d")
            if len(outside_points):
                axes.scatter(*outside_points.T, c="lightgray", s=1, alpha=0.3, label="Outside box")
            if len(inside_points):
                axes.scatter(*inside_points.T, c="crimson", s=3, alpha=0.9, label="Inside box")
            draw_box_edges(axes, box)
            axes.set_title(f"Frame {frame_number}: {len(points_to_plot):,} points, {int(inside_box.sum()):,} inside box")
            axes.set_xlabel("X (m)")
            axes.set_ylabel("Y (m)")
            axes.set_zlabel("Z (m)")
            axes.set_box_aspect(np.maximum(np.ptp(points_to_plot, axis=0), 1e-6))
            axes.legend(loc="upper right")
            plt.tight_layout()
            plt.show()
    except Exception as error:
        with single_frame_output:
            clear_output(wait=True)
            print(f"Frame preview failed: {error}")
    finally:
        single_frame_button.disabled = False


single_frame_button.on_click(plot_single_frame)
plot_single_frame(None)
display(widgets.HBox([single_frame_widget, single_frame_button]), single_frame_output)

Output()

# Plot position of best cluster centroid at each frame

Show the movement of the tracked object over time

In [19]:

def plot_centrorid_by_frame():
    frame_numbers = []
    centroids_by_frame = []

    point_filter_box = get_point_filter_box()
    box = {
        "xmin": bbox_xmin_widget.value,
        "xmax": bbox_xmax_widget.value,
        "ymin": bbox_ymin_widget.value,
        "ymax": bbox_ymax_widget.value,
        "zmin": bbox_zmin_widget.value,
        "zmax": bbox_zmax_widget.value,
    }

    for frame_number, frame in enumerate(frames[:frame_count_widget.value], start=1):
        plot_points = prepare_points(
            frame,
            min_distance_widget.value,
            max_distance_widget.value,
            keep_fraction_widget.value,
            random_seed_widget.value + frame_number - 1,
            box=point_filter_box,
        )
        if not len(plot_points):
            continue

        labels = cluster_points(
            plot_points,
            method_widget.value,
            cluster_count_widget.value,
            dbscan_eps_widget.value,
            min_samples_widget.value,
            hdbscan_min_cluster_size_widget.value,
            spectral_neighbors_widget.value,
        )
        best = best_cluster_for_box(labels, box_mask(plot_points, box))
        if best is not None and np.any(best["mask"]):
            frame_numbers.append(frame_number)
            centroids_by_frame.append(plot_points[best["mask"]].mean(axis=0))

    if centroids_by_frame:
        centroids_by_frame = np.asarray(centroids_by_frame)

        figure, axes = plt.subplots(figsize=(10, 5))
        # Position of the cetrtroid separated into x, y, and z coordinates
        #for coordinate, label in enumerate(("X", "Y", "Z")):
        #    axes.scatter(frame_numbers, centroids_by_frame[:, coordinate], label=f"{label} (m)")

        # Position of centroid as distance from scanner
        distance_from_scanner = np.sqrt(np.sum(centroids_by_frame**2, axis=1))
        axes.scatter(
            frame_numbers,
            distance_from_scanner,
            label="Distance from scanner (m)",
        )

        axes.set_title(f"Best-cluster centroid position by frame ({method_widget.value})")
        axes.set_xlabel("Frame number")
        axes.set_ylabel("Position (m)")
        axes.grid(True, alpha=0.3)
        axes.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("No best-cluster centroids found for the selected frames.")